In [1]:
import sys
print(sys.executable)
from ytclickbait.config import load_config

/Users/ina/yt-clickbait/yt-clickbait/.venv/bin/python


In [2]:
import pandas as pd
from ytclickbait.config import load_config

cfg = load_config()

videos = pd.read_parquet(cfg.file("videos"))
split = pd.read_parquet(cfg.file("split"))
rules = pd.read_parquet(cfg.file("rule_features"))

rule_cols = rules.columns.drop("video_id").tolist()

for table in (videos, split, rules):
    assert table["video_id"].notna().all()
    assert table["video_id"].is_unique

df = (
    videos
    .merge(split, on="video_id", validate="one_to_one")
    .merge(rules, on="video_id", validate="one_to_one")
)

assert len(df) == len(videos) == len(split) == len(rules)
assert len(rule_cols) == 14

print(f"Modeling Data: video {len(df):,}, rule_features {len(rule_cols)}")
display(df[["title", "log_views", "split_time", "n_exclaim", "caps_ratio"]].head())

Modeling Data: video 78,180, rule_features 14


,title,log_views,split_time,n_exclaim,caps_ratio
0,Jadon Sancho • Magical Skills & Goals,12.979280,train,0,0.172414
1,Migos - Avalanche (Official Video),15.735846,excluded,0,0.148148
2,MY TESLA PAYS FOR ITSELF,13.245354,excluded,0,1.000000
3,Supporting Actress in a Comedy: 73rd Emmys,13.433679,train,0,0.121212
4,San Francisco 49ers vs. Arizona Cardinals Game...,14.478602,test,0,0.185185


In [3]:
import pandas as pd
from ytclickbait.config import load_config

cfg = load_config()

videos = pd.read_parquet(cfg.file("videos"))
split = pd.read_parquet(cfg.file("split"))
rules = pd.read_parquet(cfg.file("rule_features"))

print("number of videos:", len(videos))
print("number of split rows:", len(split))
print("number of rule features:", len(rules.columns.drop("video_id")))

display(rules.head(3))

number of videos: 78180
number of split rows: 78180
number of rule features: 14


,video_id,n_chars,n_words,caps_ratio,n_caps_words,n_exclaim,n_question,has_number,starts_with_number,n_emoji,has_brackets,has_pipe_or_colon,has_hashtag,has_second_person,n_clickbait_words
0,--0bCF-iK2E,37,7,0.172414,0,0,0,0,0,0,0,0,0,0,0
1,--14w5SOEUs,34,5,0.148148,0,0,0,0,0,0,1,0,0,0,0
2,--2O86Z0hsM,24,5,1.000000,5,0,0,0,0,0,0,0,0,0,0


In [4]:
# split train and test

train = df.loc[df["split_time"] == "train"].copy()
test = df.loc[df["split_time"] == "test"].copy()

print(f"train: {len(train):,}")
print(f"test: {len(test):,}")
print(f"excluded: {(df['split_time'] == 'excluded').sum():,}")

train: 42,204
test: 11,487
excluded: 24,489


In [5]:
# baseline model

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = ["log_age_days", "channel_baseline_split_time"]
category_cols = ["category_name", "publish_week"]
control_cols = numeric_cols + category_cols

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_cols),
    ("category", OneHotEncoder(handle_unknown="ignore"), category_cols),
])

baseline = make_pipeline(preprocess, Ridge(alpha=1.0, solver="lsqr"))

baseline.fit(train[control_cols], train["log_views"])

baseline_pred = baseline.predict(test[control_cols])

baseline_rmse = mean_squared_error(test["log_views"], baseline_pred) ** 0.5
baseline_r2 = r2_score(test["log_views"], baseline_pred)

print(f"Baseline Model RMSE: {baseline_rmse:.4f}")
print(f"Baseline Model R²:   {baseline_r2:.4f}")

Baseline Model RMSE: 0.7385
Baseline Model R²:   0.5953


In [6]:
# numeric_cols + rule_cols

numeric_with_rules = numeric_cols + rule_cols
input_cols = numeric_with_rules + category_cols

rules_preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_with_rules),
    ("category", OneHotEncoder(handle_unknown="ignore"), category_cols),
])

rules_model = make_pipeline(rules_preprocess, Ridge(alpha=1.0, solver="lsqr"))
rules_model.fit(train[input_cols], train["log_views"])

rules_pred = rules_model.predict(test[input_cols])
rules_rmse = mean_squared_error(test["log_views"], rules_pred) ** 0.5
rules_r2 = r2_score(test["log_views"], rules_pred)

results = pd.DataFrame({
    "Model": ["numeric_cols", "numeric_cols + rule_cols_14"],
    "RMSE": [baseline_rmse, rules_rmse],
    "R²": [baseline_r2, rules_r2],
})
display(results.round(4))

,Model,RMSE,R²
0,numeric_cols,0.7385,0.5953
1,numeric_cols + rule_cols_14,0.7350,0.5992


In [ ]:
# 1. train & test
group_train = df.loc[df["split_group"] == "train"]
group_test = df.loc[df["split_group"] == "test"]

# Check if test dataset not in train dataset
assert set(group_train["channel_id"]).isdisjoint(group_test["channel_id"])

# 2. Compare 
group_controls = ["log_age_days", "channel_baseline_split_group"]
group_scores = {}

for name, extra_cols in [("control_variables", []), ("control_variables + rule_based_features", rule_cols)]:
    num_cols = group_controls + extra_cols
    x_cols = num_cols + category_cols

    preprocess_group = ColumnTransformer([
        ("numeric", StandardScaler(), num_cols),
        ("category", OneHotEncoder(handle_unknown="ignore"), category_cols),
    ])
    model = make_pipeline(preprocess_group, Ridge(alpha=1.0, solver="lsqr"))
    model.fit(group_train[x_cols], group_train["log_views"])

    pred = model.predict(group_test[x_cols])
    group_scores[name] = {
        "RMSE": mean_squared_error(group_test["log_views"], pred) ** 0.5,
        "R²": r2_score(group_test["log_views"], pred),
    }

display(pd.DataFrame(group_scores).T.round(4))

,RMSE,R²
control_variables,1.2048,0.1180
control_variables + rule_based_features,1.2029,0.1207


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = df.loc[df["split_time"] == "train"]
test = df.loc[df["split_time"] == "test"]

numeric_controls = ["log_age_days", "channel_baseline_split_time"]
category_cols = ["category_name", "publish_week"]

rows = []

for name, extra_cols in [
    ("Control Variables", []),
    ("Control Variables + Rule Features", rule_cols),
]:
    numeric_cols = numeric_controls + extra_cols
    input_cols = numeric_cols + category_cols

    preprocess = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_cols),
        ("category", OneHotEncoder(handle_unknown="ignore"), category_cols),
    ])
    model = make_pipeline(preprocess, Ridge(alpha=1.0, solver="lsqr"))
    model.fit(train[input_cols], train["log_views"])

    pred = model.predict(test[input_cols])
    rows.append({
        "Model": name,
        "RMSE": mean_squared_error(test["log_views"], pred) ** 0.5,
        "R²": r2_score(test["log_views"], pred),
    })

results = pd.DataFrame(rows)
display(pd.DataFrame(results).round(4))

,Model,RMSE,R²
0,Control Variables,0.7385,0.5953
1,Control Variables + Rule Features,0.7350,0.5992
